
# Expiry Risk Detection Engine

This notebook implements a **Rules-Based Expiry Risk Detection System** for pharmaceutical inventory.

### Goal
Identify batches nearing expiration and estimate potential financial loss.

### Risk Levels
- Critical → <= 30 days
- High → <= 60 days
- Medium → <= 120 days
- Low → > 120 days


In [ ]:

!pip install pandas matplotlib


In [ ]:

import pandas as pd
from datetime import datetime



## Upload Dataset

Upload the dataset:
pioneer_pharma_ideal_dataset_70000_rows.csv



In [ ]:

from google.colab import files
uploaded = files.upload()


## Load Dataset

In [ ]:

df = pd.read_csv("pioneer_pharma_ideal_dataset_70000_rows.csv")

df.head()


## Convert Date Columns

In [ ]:

df["expiry_date"] = pd.to_datetime(df["expiry_date"])
df["order_date"] = pd.to_datetime(df["order_date"])


## Calculate Days to Expiry

In [ ]:

today = pd.Timestamp(datetime.today())

df["days_to_expiry"] = (df["expiry_date"] - today).dt.days

df.head()


## Risk Scoring Rules

In [ ]:

def calculate_risk(days):

    if days <= 30:
        return "Critical"

    elif days <= 60:
        return "High"

    elif days <= 120:
        return "Medium"

    else:
        return "Low"


## Apply Risk Levels

In [ ]:

df["risk_level"] = df["days_to_expiry"].apply(calculate_risk)

df.head()


## Calculate Potential Financial Loss

In [ ]:

df["potential_value_loss"] = df["quantity"] * df["unit_price_iqd"]

df.head()


## Identify High Risk Inventory

In [ ]:

high_risk = df[df["risk_level"].isin(["Critical","High"])]

high_risk.head()


## Sort by Financial Risk

In [ ]:

high_risk_sorted = high_risk.sort_values(
    by="potential_value_loss",
    ascending=False
)

high_risk_sorted.head(20)


## Risk Summary

In [ ]:

risk_summary = df.groupby("risk_level")["potential_value_loss"].sum()

risk_summary


## Risk Visualization

In [ ]:

import matplotlib.pyplot as plt

risk_summary.plot(kind="bar")

plt.title("Financial Exposure by Expiry Risk Level")
plt.ylabel("Potential Loss (IQD)")

plt.show()
